# Generate Draf Tanya-Jawab dari Pedoman Akademik (Ollama di Google Colab)

Notebook ini membuat **draf** pasangan tanya-jawab dari `chunks.jsonl` (hasil `scripts/extract_pdf.py`) memakai **Qwen 2.5 14B** yang dijalankan dengan Ollama di GPU Colab. Di Colab prosesnya jauh lebih cepat daripada di CPU laptop.

**Sebelum mulai:**
1. `Runtime → Change runtime type → T4 GPU`.
2. Upload `data/processed/chunks.jsonl` ke Google Drive: `MyDrive/proyek-llm/data/processed/chunks.jsonl` (struktur folder Drive = struktur folder proyek di laptop).
3. Jalankan sel dari atas ke bawah. Hasil disimpan **bertahap** ke `MyDrive/proyek-llm/data/raw/qa_draft.csv`. Kalau sesi terputus, jalankan ulang semua sel dan proses akan **melanjutkan** dari chunk terakhir.

**Setelah selesai:** download `qa_draft.csv` ke `D:\proyek-llm\data\raw\`, lalu **review manual** setiap baris (isi `status` = `ok`) sebelum menjalankan `scripts/prepare_dataset.py`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR    = '/content/drive/MyDrive/proyek-llm/data'
CHUNKS_PATH = f'{DATA_DIR}/processed/chunks.jsonl'
OUT_PATH    = f'{DATA_DIR}/raw/qa_draft.csv'
assert os.path.exists(CHUNKS_PATH), f'Upload dulu chunks.jsonl ke {CHUNKS_PATH}'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install & jalankan Ollama

In [ ]:
%%capture
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(['ollama', 'serve'],
                               stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://127.0.0.1:11434/api/version', timeout=2)
        print('Ollama server jalan')
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Ollama gagal start, cek /content/ollama.log')

## 3. Download model generator
`qwen2.5:14b` (~9 GB) muat di VRAM T4 dan lebih akurat. Kalau GPU-nya lebih kecil atau ingin lebih cepat, ganti ke `qwen2.5:7b`.

In [ ]:
MODEL = 'qwen2.5:14b'
!ollama pull {MODEL}

## 4. Prompt & fungsi generate

In [ ]:
import json, re

PROMPT = '''Berikut potongan Pedoman Akademik Universitas Telkom:

=== AWAL TEKS ===
{teks}
=== AKHIR TEKS ===

Buat {n} pasangan tanya-jawab dalam bahasa Indonesia yang mungkin ditanyakan mahasiswa,
HANYA berdasarkan isi teks di atas (jangan mengarang informasi yang tidak ada).
- Variasikan gaya pertanyaan: formal, santai, singkat, ada yang memakai kata "gimana", "boleh gak", "kak", dll.
- Jawaban 1-4 kalimat, sopan, jelas, sebutkan angka/syarat persis seperti di teks.
- Jawaban ditulis seperti petugas layanan akademik yang menjawab langsung. JANGAN menulis
  "berdasarkan teks di atas" atau "menurut potongan tersebut".
- Jika aturan berbeda per jenjang (diploma, sarjana, magister, doktor), utamakan program sarjana
  dan sebutkan jenjangnya di jawaban.
- Isi "kategori" dengan topik singkat (misal: krs, cuti akademik, masa studi, nilai, tugas akhir, kelulusan, sanksi).

Keluarkan HANYA JSON dengan format:
{{"qa": [{{"pertanyaan": "...", "jawaban": "...", "kategori": "..."}}]}}'''

def jumlah_qa(teks):
    # chunk pendek -> sedikit QA, chunk panjang -> lebih banyak (3..10)
    return max(3, min(10, len(teks) // 250))

def generate(teks, n, coba=2):
    for k in range(coba):
        r = requests.post('http://127.0.0.1:11434/api/chat', timeout=900, json={
            'model': MODEL, 'stream': False, 'format': 'json',
            'messages': [{'role': 'user', 'content': PROMPT.format(teks=teks, n=n)}],
            'options': {'temperature': 0.7, 'num_ctx': 8192},
        })
        r.raise_for_status()
        try:
            items = json.loads(r.json()['message']['content']).get('qa', [])
            items = [q for q in items if isinstance(q, dict) and q.get('pertanyaan') and q.get('jawaban')]
            if items:
                return items
        except json.JSONDecodeError:
            pass
        print(f'   JSON tidak valid, coba lagi ({k + 1}/{coba})')
    return []

chunks = [json.loads(l) for l in open(CHUNKS_PATH, encoding='utf-8')]
print(f'{len(chunks)} chunk, perkiraan total QA: {sum(jumlah_qa(c["teks"]) for c in chunks)}')

## 5. Uji 1 chunk dulu
Cek kualitasnya. Kalau jawabannya mengarang atau formatnya aneh, perbaiki `PROMPT` sebelum lanjut.

In [ ]:
contoh = next(c for c in chunks if c.get('pasal') == 57) if any(c.get('pasal') == 57 for c in chunks) else chunks[0]
t0 = time.time()
for qa in generate(contoh['teks'], 5):
    print('Q:', qa['pertanyaan']); print('A:', qa['jawaban']); print('  [', qa.get('kategori'), ']\n')
print(f'{time.time() - t0:.0f} detik')

## 6. Generate semua chunk (bisa dilanjutkan bila terputus)

In [ ]:
import csv
from datetime import datetime

KOLOM = ['pertanyaan', 'jawaban', 'kategori', 'sumber', 'status', 'chunk_id']

# Lanjutkan: cari chunk_id yang sudah pernah diproses
sudah = set()
if os.path.exists(OUT_PATH):
    with open(OUT_PATH, encoding='utf-8-sig', newline='') as f:
        sudah = {int(r['chunk_id']) for r in csv.DictReader(f) if r.get('chunk_id')}
    print(f'Melanjutkan: {len(sudah)} chunk sudah selesai')

baru = not os.path.exists(OUT_PATH)
t_mulai = time.time()
with open(OUT_PATH, 'a', encoding='utf-8-sig' if baru else 'utf-8', newline='') as fo:
    w = csv.writer(fo)
    if baru:
        w.writerow(KOLOM)
    for i, c in enumerate(chunks):
        if i in sudah:
            continue
        t0 = time.time()
        try:
            items = generate(c['teks'], jumlah_qa(c['teks']))
        except Exception as e:
            print(f'[{i + 1}/{len(chunks)}] GAGAL: {e}')
            continue
        for qa in items:
            w.writerow([qa['pertanyaan'].strip(), qa['jawaban'].strip(),
                        str(qa.get('kategori', '')).strip().lower(),
                        '; '.join(c['sumber']), 'draft', i])
        fo.flush()
        print(f'[{i + 1}/{len(chunks)}] Pasal {c.get("pasal", "?")}: +{len(items)} QA ({time.time() - t0:.0f} s)')

print(f'Selesai dalam {(time.time() - t_mulai) / 60:.1f} menit -> {OUT_PATH}')

## 7. Ringkasan hasil

In [ ]:
import pandas as pd
df = pd.read_csv(OUT_PATH, encoding='utf-8-sig')
print('Total draf QA :', len(df))
print('Chunk berhasil:', df['chunk_id'].nunique(), '/', len(chunks))
print('Chunk tanpa QA:', sorted(set(range(len(chunks))) - set(df['chunk_id'])))
print('Duplikat pertanyaan:', df['pertanyaan'].str.lower().duplicated().sum())
display(df['kategori'].value_counts().head(20))
df.sample(min(10, len(df)), random_state=1)[['pertanyaan', 'jawaban', 'sumber']]

## 8. Download
File juga sudah ada di Drive (`MyDrive/proyek-llm/data/raw/qa_draft.csv`).

In [ ]:
from google.colab import files
files.download(OUT_PATH)

## Langkah berikutnya (di laptop)
1. Simpan `qa_draft.csv` ke `D:\proyek-llm\data\raw\`.
2. Review di Excel/Google Sheets: cocokkan jawaban dengan pasal di kolom `sumber`, perbaiki yang salah, hapus yang tidak relevan, isi `status` = `ok`.
3. `python scripts/prepare_dataset.py data/raw/qa_draft.csv data/raw/qa_manual.csv`